# Pipelines and Functional Operations

The neuroimpy package provides a set of functions that allow you to split image data in various ways for processing. By breaking a dataset into pieces, operations can be parallelized and complex analyses can be composed from simple building blocks.

## Splitting an image into connected components

First, let's find connected components in a thresholded volume

In [ ]:
# Import required libraries
import neuroimpy as pn
import numpy as np
import matplotlib.pyplot as plt

# Create example data for tutorial
# (In practice, you would use real neuroimaging files)

# Create example volume
space_3d = pn.NeuroSpace(dim=(64, 64, 25), spacing=(3.5, 3.5, 5.0), 
                         origin=(-110.5, -88.9342, -42.75))
example_data = np.random.randn(64, 64, 25)
example_data = (example_data - example_data.min()) / (example_data.max() - example_data.min())

# Create volume
vol = pn.DenseNeuroVol(example_data, space_3d)

# Create another volume with higher values in some regions
vol2 = vol.as_dense()
# Add some high-value regions
vol2.data[20:30, 20:30, 10:15] = 0.9
vol2.data[40:50, 40:50, 15:20] = 0.85

# Find connected components with threshold of 0.8
comp = pn.conn_comp(vol2, threshold=0.8)

# comp is a ConnCompResult with:
# - comp.index: ClusteredNeuroVol with component labels
# - comp.size: component sizes per voxel
# - comp.voxels: voxel coordinates for each cluster

print(f"Found {comp.index.num_clusters} connected components")
if comp.cluster_table is not None:
    print(f"Largest component has {comp.cluster_table['size'].max()} voxels")
else:
    print(f"Component sizes stored in comp.size volume")

## Computing statistics on connected components

In [ ]:
# Calculate mean value in each connected component
from neuroimpy import split_clusters

cluster_rois = split_clusters(vol2, comp.index)

# Compute mean for each cluster
means = [np.mean(roi.data) for roi in cluster_rois]
print(f"Component means: {means[:5]}...")  # Show first 5

## Searchlight analysis

In [ ]:
# Create searchlights with 5mm radius
from neuroimpy import searchlight

# Create a mask
mask = vol.data > 0.2
mask_vol = pn.LogicalNeuroVol(mask, vol.space)

# Get searchlight ROIs (limit to first 100 for demo)
rois = list(searchlight(mask_vol, radius=5))[:100]

# Compute standard deviation in each
sd_values = []
indices = []

for roi in rois:
    # Extract values from the volume
    values = vol2[roi.coords[:, 0], roi.coords[:, 1], roi.coords[:, 2]]
    sd_values.append(np.std(values))
    
    # Get center voxel index
    center_coord = roi.coords[len(roi)//2]  # Approximate center
    indices.append(vol.space.grid_to_index(center_coord.reshape(1, -1))[0])

# Create output volume
sd_vol = pn.SparseNeuroVol(
    data=np.array(sd_values),
    space=vol.space,
    indices=np.array(indices)
)

print(f"Computed local SD for {len(sd_values)} voxels")

## K-nearest neighbors smoothing

In [ ]:
k = 12
smoothed_values = []
indices = []

# Process first 50 searchlights for demo
mask_vol = pn.LogicalNeuroVol(mask, vol2.space)
for roi in list(searchlight(mask_vol, radius=6))[:50]:
    # Get values in the ROI
    roi_coords = roi.coords
    roi_values = vol2[roi_coords[:, 0], roi_coords[:, 1], roi_coords[:, 2]]
    
    # Assume center is middle of ROI
    center_idx = len(roi_values) // 2
    center_val = roi_values[center_idx]
    
    # Find k nearest neighbors by value
    distances = np.abs(roi_values - center_val)
    k_nearest_idx = np.argsort(distances)[1:k+1]  # Exclude center itself
    smoothed_values.append(np.mean(roi_values[k_nearest_idx]))
    
    # Store center index
    center_coord = roi_coords[center_idx]
    indices.append(vol.space.grid_to_index(center_coord.reshape(1, -1))[0])

# Create smoothed volume
smoothed_vol = pn.SparseNeuroVol(
    data=np.array(smoothed_values),
    space=vol2.space,
    indices=np.array(indices)
)

print(f"Smoothed {len(smoothed_values)} voxels")

## Using searchlight coordinates only

In [ ]:
from neuroimpy import searchlight_coords

# Get coordinates for each searchlight
avg_values = []
center_indices = []

# Process first 50 searchlights
for coords in list(searchlight_coords(mask_vol, radius=12))[:50]:
    # coords is an Nx3 array of voxel coordinates
    vals = vol.data[coords[:, 0], coords[:, 1], coords[:, 2]]
    
    # Average only non-zero values
    nonzero_vals = vals[vals != 0]
    if len(nonzero_vals) > 0:
        avg_values.append(np.mean(nonzero_vals))
        # Assume first coordinate is center
        center_indices.append(vol.space.grid_to_index(coords[0:1])[0])

# Create averaged volume
avg_vol = pn.SparseNeuroVol(
    data=np.array(avg_values),
    space=vol.space,
    indices=np.array(center_indices)
)

print(f"Created averaged volume with {len(avg_values)} voxels")

## Processing slices

In [ ]:
# Process each axial slice
slice_means = []
for z in range(vol.shape[2]):
    slice_data = vol.data[:, :, z]
    slice_means.append(np.mean(slice_data))

# Plot slice means
plt.figure(figsize=(10, 4))
plt.plot(slice_means)
plt.xlabel('Slice number')
plt.ylabel('Mean intensity')
plt.title('Mean intensity by slice')
plt.show()

## Processing volumes in a 4D dataset

In [ ]:
# Create a 4D dataset
space_4d = pn.NeuroSpace(dim=(10, 10, 10, 5))
data_4d = np.random.randn(10, 10, 10, 5)
vec = pn.DenseNeuroVec(data_4d, space_4d)

print(f"4D data shape: {vec.shape}")

# Calculate statistics for each volume
volumes = vec.vols()  # Get list of all volumes
mean_vec = [np.mean(v.data) for v in volumes]
sd_vec = [np.std(v.data) for v in volumes]

print(f"Number of volumes: {len(mean_vec)}")
print(f"Mean values: {mean_vec}")
print(f"SD values: {sd_vec}")

## Processing voxel time series

In [ ]:
# Calculate mean across time for each voxel
space_4d = pn.NeuroSpace(dim=(10, 10, 10, 50))
vec = pn.DenseNeuroVec(np.random.randn(10, 10, 10, 50), space_4d)

# Method 1: Using numpy operations
mean_vol_data = np.mean(vec.data, axis=3)
mean_vol = pn.DenseNeuroVol(mean_vol_data, vec.space.sub_space())

# Method 2: Using voxel iteration
mean_values = []
for i in range(vec.shape[0]):
    for j in range(vec.shape[1]):
        for k in range(vec.shape[2]):
            ts = vec.series(i, j, k)
            mean_values.append(np.mean(ts))

# Reshape to volume
mean_vol2_data = np.array(mean_values).reshape(vec.shape[:3], order='F')
mean_vol2 = pn.DenseNeuroVol(mean_vol2_data, mean_vol.space)

print(f"Mean volume shape: {mean_vol.shape}")

## Parallel processing

In [ ]:
from joblib import Parallel, delayed

def process_roi(roi, volume):
    """Process a single ROI"""
    values = volume[roi.coords[:, 0], roi.coords[:, 1], roi.coords[:, 2]]
    return {
        'mean': np.mean(values),
        'std': np.std(values),
        'max': np.max(values),
        'size': len(values)
    }

# Create ROIs (limit for demo)
rois = list(searchlight(mask_vol, radius=8))[:20]

# Process in parallel
results = Parallel(n_jobs=-1)(
    delayed(process_roi)(roi, vol2) for roi in rois
)

# Extract results
roi_means = [r['mean'] for r in results]
roi_stds = [r['std'] for r in results]

print(f"Processed {len(results)} ROIs in parallel")
print(f"Mean values range: [{min(roi_means):.3f}, {max(roi_means):.3f}]")

## Building analysis pipelines

In [ ]:
def denoise_pipeline(vol, smooth_radius=5, threshold=0.1):
    """
    Example denoising pipeline:
    1. Identify noise components
    2. Smooth within components
    3. Threshold final result
    """
    # Find connected components above threshold
    comp = pn.conn_comp(vol, threshold=threshold)
    
    # Process each component
    processed_rois = []
    for roi in split_clusters(vol, comp.index):
        if len(roi) < 10:  # Skip small components
            continue
            
        # Smooth within component
        roi_coords = roi.coords
        values = vol[roi_coords[:, 0], roi_coords[:, 1], roi_coords[:, 2]]
        
        # Simple smoothing: multiply by factor
        smoothed_values = values * 0.8
        
        # Update ROI with smoothed values
        roi.data = smoothed_values
        processed_rois.append(roi)
    
    # Combine back into volume
    result = pn.DenseNeuroVol(np.zeros_like(vol.data), vol.space)
    for roi in processed_rois:
        result[roi.coords[:, 0], roi.coords[:, 1], roi.coords[:, 2]] = roi.data
    
    return result

# Apply pipeline
denoised = denoise_pipeline(vol2, smooth_radius=5, threshold=0.2)
print(f"Denoised volume range: [{denoised.min():.3f}, {denoised.max():.3f}]")

## Custom iterators

In [ ]:
def sliding_window_iterator(vec, window_size=10, step=5):
    """
    Iterate over overlapping time windows
    """
    n_timepoints = vec.shape[3]
    for start in range(0, n_timepoints - window_size + 1, step):
        end = start + window_size
        yield vec.sub_vector(slice(start, end))

# Create 4D data
space_4d = pn.NeuroSpace(dim=(10, 10, 10, 100))
vec = pn.DenseNeuroVec(np.random.randn(10, 10, 10, 100), space_4d)

# Use the iterator
window_means = []
for window_vec in sliding_window_iterator(vec, window_size=20, step=10):
    # Calculate mean volume for this time window
    mean_vol = np.mean(window_vec.data, axis=3)
    window_means.append(mean_vol)

print(f"Computed {len(window_means)} windowed mean volumes")
print(f"Each window mean volume has shape: {window_means[0].shape}")